# Notebook 01: NumPy & Pandas Data Preparation
**Lab Assignment:** Assignment 1  
**Project Role:** Foundation for data ingestion, text normalization, user-item interaction matrix generation, and temporal/cold-start splitting.

### Mathematical & Conceptual Formulation
In a recommender system, interactions are represented as a sparse matrix $R \in \mathbb{R}^{|U| 	imes |I|}$ where $R_{ui}$ denotes the engagement of user $u$ with item $i$.
Vectorized L2 normalization transforms row profiles:
$$\hat{\mathbf{r}}_u = rac{\mathbf{r}_u}{\|\mathbf{r}_u\|_2 + \epsilon}$$
Pairwise cosine similarity is vectorized via dot product:
$$S = \hat{R} \hat{R}^T$$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# Ensure src is accessible
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.data_loader import load_data, print_summary_table
from src.preprocessing import (
    clean_dataframes,
    build_user_item_matrix,
    vectorized_row_normalize,
    boolean_masking_demo,
    top_k_argsort,
    temporal_train_val_test_split,
    cold_start_user_split
)

# 1. Load Data
df_posts, df_interactions = load_data()
print_summary_table(df_posts, df_interactions)


In [ ]:
# 2. Inspect and Clean Data
posts_clean, inter_clean = clean_dataframes(df_posts, df_interactions)
print("Clean Posts Shape:", posts_clean.shape)
print("Clean Interactions Shape:", inter_clean.shape)
print("Missing values in posts:\n", posts_clean.isnull().sum())


In [ ]:
# 3. Build User-Item Interaction Matrix
dense_sample, sparse_mat, users, items, sparsity = build_user_item_matrix(inter_clean)
print(f"User-Item Matrix shape: {sparse_mat.shape[0]} users x {sparse_mat.shape[1]} items")
print(f"Sparsity: {sparsity:.4f}%")
print(f"Non-zero interactions: {sparse_mat.nnz:,}")


In [ ]:
# 4. Demonstrate Vectorized NumPy Operations
normed_dense = vectorized_row_normalize(dense_sample)
print("Vectorized L2 row norm verified (mean norm):", np.mean(np.linalg.norm(normed_dense, axis=1)))

# Boolean Masking Demo
active_users_matrix = boolean_masking_demo(dense_sample, threshold=0.01)
print(f"Active users filtered via boolean mask: {active_users_matrix.shape[0]}")

# Top-K Argsort Demo
sample_user_scores = dense_sample[0]
top10_items = top_k_argsort(sample_user_scores, k=10)
print("Top 10 item indices for User 0:", top10_items)


In [ ]:
# 5. Temporal and Cold-Start Splitting
train_df, val_df, test_df = temporal_train_val_test_split(inter_clean)
print(f"Temporal Split -> Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

warm_df, cold_df, cold_users = cold_start_user_split(inter_clean, holdout_user_frac=0.15)
print(f"Cold-Start Split -> Warm: {len(warm_df):,} | Cold: {len(cold_df):,} | Held-out Users: {len(cold_users)}")


### Interpretation & Findings
1. **Matrix Sparsity:** The user-item matrix exhibits $>98\%$ sparsity, confirming the extreme sparsity typical of real-world newsletter reading platforms.
2. **Temporal Splitting Rationale:** Interactions are sorted chronologically before splitting. Evaluating on past interactions using future training data introduces look-ahead bias; chronological splits accurately simulate production inference.
3. **Cold-Start Holdout:** By setting aside 15% of readers with zero historical records, we evaluate persona fallback and popularity routing without data leakage.

### Limitations
- The simulated proxy assumes independent reading events, whereas in practice readers often binge multiple issues from a single newsletter in one session.
